In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# 1. Load Dataset (Glass Identification)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/glass/glass.data"
column_names = ['Id', 'RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'Type']
df = pd.read_csv(url, names=column_names)
df = df.drop('Id', axis=1) # ID is not a feature

# 2. Preprocessing
X = df.drop('Type', axis=1).values
y = df['Type'].values

# Labels are 1-7, we need them to be 0-5 for matrix indexing
le = LabelEncoder()
y = le.fit_transform(y)

# Feature Scaling is crucial for Perceptrons
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Single Layer Perceptron Implementation
class SLPClassifier:
    def __init__(self, learning_rate=0.01, epochs=1000):
        self.lr = learning_rate
        self.epochs = epochs

    def softmax(self, z):
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def fit(self, X, y):
        n_samples, n_features = X.shape
        n_classes = len(np.unique(y))
        self.weights = np.random.randn(n_features, n_classes) * 0.01
        self.bias = np.zeros(n_classes)

        y_onehot = np.eye(n_classes)[y]

        for i in range(self.epochs):
            # Forward
            scores = np.dot(X, self.weights) + self.bias
            probs = self.softmax(scores)

            # Gradient Descent
            dz = probs - y_onehot
            dw = np.dot(X.T, dz) / n_samples
            db = np.sum(dz, axis=0) / n_samples

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

    def predict(self, X):
        scores = np.dot(X, self.weights) + self.bias
        return np.argmax(self.softmax(scores), axis=1)

# 4. Train and Evaluate
model = SLPClassifier(learning_rate=0.1, epochs=2000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 72.09%

Classification Report:
               precision    recall  f1-score   support

           0       0.69      0.82      0.75        11
           1       0.62      0.71      0.67        14
           2       0.00      0.00      0.00         3
           3       1.00      0.50      0.67         4
           4       1.00      0.67      0.80         3
           5       0.80      1.00      0.89         8

    accuracy                           0.72        43
   macro avg       0.69      0.62      0.63        43
weighted avg       0.69      0.72      0.69        43



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
